In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from imblearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, recall_score, make_scorer, precision_recall_curve,
    roc_auc_score, average_precision_score, cohen_kappa_score, PrecisionRecallDisplay
)
from imblearn.over_sampling import SMOTE
from collections import Counter
from pytorch_tabnet.tab_model import TabNetClassifier
import torch
import optuna
from sklearn.base import BaseEstimator, TransformerMixin

## Data preparation

In [2]:
# =====================
# 1. Prepare Data
# =====================

# Load data
df = pd.read_csv(r"C:\Users\dbastola2022\OneDrive - Florida Atlantic University\Academics\Research\Malnutrition\MICS\malnutrition\Dataset\ch.csv") #Local

# Split features and target
X = df.drop(columns=['status'])
y = df['status']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Apply SMOTE to training data only
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)

# Convert to numpy arrays for TabNet
X_train_sm_np = X_train_sm.to_numpy().astype(np.float32)
X_test_np = X_test.to_numpy().astype(np.float32)
y_train_sm_np = y_train_sm.to_numpy().astype(np.int64)
y_test_np = y_test.to_numpy().astype(np.int64)

## Base Model

In [3]:
# ----------------------------
# 2. Base TabNet Model
# ----------------------------
base_tabnet = TabNetClassifier(
    seed=42,
    verbose=1,
    device_name='cuda' if torch.cuda.is_available() else 'cpu'
)

base_tabnet.fit(
    X_train_sm_np, y_train_sm_np,
    eval_set=[(X_test_np, y_test_np)],
    eval_name=["val"],
    eval_metric=["balanced_accuracy"],
    max_epochs=100,
    patience=10,
    batch_size=1024,
    virtual_batch_size=128,
    num_workers=0
)

c:\Users\dbastola2022\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.93647 | val_balanced_accuracy: 0.43682 |  0:00:00s
epoch 1  | loss: 0.70055 | val_balanced_accuracy: 0.44297 |  0:00:00s
epoch 2  | loss: 0.68066 | val_balanced_accuracy: 0.57021 |  0:00:00s
epoch 3  | loss: 0.66007 | val_balanced_accuracy: 0.62049 |  0:00:01s
epoch 4  | loss: 0.65168 | val_balanced_accuracy: 0.62758 |  0:00:01s
epoch 5  | loss: 0.65009 | val_balanced_accuracy: 0.6406  |  0:00:01s
epoch 6  | loss: 0.64423 | val_balanced_accuracy: 0.63075 |  0:00:01s
epoch 7  | loss: 0.63826 | val_balanced_accuracy: 0.64344 |  0:00:02s
epoch 8  | loss: 0.63307 | val_balanced_accuracy: 0.65873 |  0:00:02s
epoch 9  | loss: 0.63017 | val_balanced_accuracy: 0.64847 |  0:00:02s
epoch 10 | loss: 0.62348 | val_balanced_accuracy: 0.6363  |  0:00:03s
epoch 11 | loss: 0.61729 | val_balanced_accuracy: 0.63831 |  0:00:03s
epoch 12 | loss: 0.61129 | val_balanced_accuracy: 0.63137 |  0:00:03s
epoch 13 | loss: 0.61786 | val_balanced_accuracy: 0.63491 |  0:00:03s
epoch 14 | loss: 0.6

c:\Users\dbastola2022\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [4]:
y_pred = base_tabnet.predict(X_test_np)
y_proba = base_tabnet.predict_proba(X_test_np)[:, 1]
y_train_pred = base_tabnet.predict(X_train_sm_np)

print("\n--- TabNet Model Performance ---")
print(confusion_matrix(y_test_np, y_pred))
print(classification_report(y_test_np, y_pred, digits=3))
print('------------------------------------------------------')
print(f"Test Average Precision: {average_precision_score(y_test, y_proba):.3f}")
print(f"Test AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"Test Cohen's Kappa: {cohen_kappa_score(y_test, y_pred):.3f}")
print(f"Training recall: {recall_score(y_train_sm_np, y_train_pred):.3f}")



--- TabNet Model Performance ---
[[525 182]
 [153 451]]
              precision    recall  f1-score   support

           0      0.774     0.743     0.758       707
           1      0.712     0.747     0.729       604

    accuracy                          0.744      1311
   macro avg      0.743     0.745     0.744      1311
weighted avg      0.746     0.744     0.745      1311

------------------------------------------------------
Test Average Precision: 0.813
Test AUC: 0.815
Test Cohen's Kappa: 0.488
Training recall: 0.740


## Tuned Model

## First

In [5]:
# import numpy as np
# import pandas as pd
# from sklearn.model_selection import train_test_split, RandomizedSearchCV
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import classification_report
# from sklearn.base import BaseEstimator, ClassifierMixin
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as ImbPipeline
# from pytorch_tabnet.tab_model import TabNetClassifier
# import torch

# # === Load Data ===
# df = pd.read_csv("C:/Users/dbastola2022/OneDrive - Florida Atlantic University/Academics/Research/Malnutrition/MICS/malnutrition/Dataset/ch.csv")

# X = df.drop(columns=['status'])
# y = df['status']

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42
# )

# # === TabNet Wrapper ===
# class TabNetWrapper(BaseEstimator, ClassifierMixin):
#     def __init__(self, n_d=8, n_a=8, n_steps=3, gamma=1.3, lambda_sparse=0.001,
#                  optimizer_fn=torch.optim.Adam, optimizer_params=None,
#                  mask_type='sparsemax', lr=2e-2, batch_size=256, virtual_batch_size=128):
#         self.n_d = n_d
#         self.n_a = n_a
#         self.n_steps = n_steps
#         self.gamma = gamma
#         self.lambda_sparse = lambda_sparse
#         self.optimizer_fn = optimizer_fn
#         self.optimizer_params = optimizer_params or {"lr": lr}
#         self.mask_type = mask_type
#         self.lr = lr
#         self.batch_size = batch_size
#         self.virtual_batch_size = virtual_batch_size
#         self.model = None

#     def fit(self, X, y):
#         X_np = X.values.astype(np.float32) if isinstance(X, pd.DataFrame) else X.astype(np.float32)
#         y_np = y.values.astype(int) if isinstance(y, pd.Series) else y.astype(int)

#         self.model = TabNetClassifier(
#             n_d=self.n_d, n_a=self.n_a, n_steps=self.n_steps, gamma=self.gamma,
#             lambda_sparse=self.lambda_sparse, optimizer_fn=self.optimizer_fn,
#             optimizer_params=self.optimizer_params, mask_type=self.mask_type,
#             verbose=0
#         )

#         self.model.fit(
#             X_np,
#             y_np,
#             eval_set=[(X_np, y_np)],
#             batch_size=self.batch_size,
#             virtual_batch_size=self.virtual_batch_size,
#             max_epochs=100,
#             patience=10
#         )
#         return self

#     def predict(self, X):
#         X_np = X.values.astype(np.float32) if isinstance(X, pd.DataFrame) else X.astype(np.float32)
#         return self.model.predict(X_np)

#     def predict_proba(self, X):
#         X_np = X.values.astype(np.float32) if isinstance(X, pd.DataFrame) else X.astype(np.float32)
#         return self.model.predict_proba(X_np)


# # === Pipeline ===
# pipeline = ImbPipeline([
#     ('smote', SMOTE(random_state=42)),
#     ('tabnet', TabNetWrapper())
# ])

# # === Hyperparameter Search Space ===
# param_dist = {
#     'tabnet__n_d': [8, 16, 32],
#     'tabnet__n_a': [8, 16, 32],
#     'tabnet__n_steps': [3, 4, 5],
#     'tabnet__gamma': [1.0, 1.3, 1.5],
#     'tabnet__lambda_sparse': [1e-3, 1e-4],
#     'tabnet__optimizer_params': [{'lr': 0.02}, {'lr': 0.01}, {'lr': 0.005}],
#     'tabnet__batch_size': [128, 256],
#     'tabnet__virtual_batch_size': [64, 128]
# }

# # === Randomized Search ===
# search = RandomizedSearchCV(
#     estimator=pipeline,
#     param_distributions=param_dist,
#     n_iter=10,
#     cv=3,
#     verbose=1,
#     scoring='recall',
#     random_state=42
# )

# # === Fit Search ===
# search.fit(X_train, y_train)

# # === Evaluation ===
# y_pred = search.predict(X_test)
# print(classification_report(y_test, y_pred))

# # Best parameters
# print("Best Parameters:\n", search.best_params_)


In [6]:
# # SMOTE (if you haven't already applied)
# sm = SMOTE(random_state=42)
# X_resampled, y_resampled = sm.fit_resample(X_train, y_train)

# X_resampled = X_resampled.values if hasattr(X_resampled, "values") else X_resampled
# y_resampled = y_resampled.values if hasattr(y_resampled, "values") else y_resampled
# X_test_np = X_test.values if hasattr(X_test, "values") else X_test
# y_test_np = y_test.values if hasattr(y_test, "values") else y_test

# def objective(trial):
#     params = {
#         "n_d": trial.suggest_int("n_d", 8, 64),
#         "n_a": trial.suggest_int("n_a", 8, 64),
#         "n_steps": trial.suggest_int("n_steps", 3, 10),
#         "gamma": trial.suggest_float("gamma", 1.0, 2.0),
#         "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
#         "optimizer_params": dict(lr=trial.suggest_float("lr", 1e-4, 1e-2, log=True)),
#         "mask_type": trial.suggest_categorical("mask_type", ["entmax", "sparsemax"]),
#         "seed": 42,
#         "verbose": 0,
#     }

#     model = TabNetClassifier(**params, device_name='cuda' if torch.cuda.is_available() else 'cpu')

#     model.fit(
#         X_resampled, y_resampled,
#         eval_set=[(X_test_np, y_test_np)],
#         eval_metric=['auc'],
#         max_epochs=100,
#         patience=15,
#         batch_size=1024,
#         virtual_batch_size=128,
#         drop_last=False
#     )

#     preds_proba = model.predict_proba(X_test_np)[:, 1]
#     return roc_auc_score(y_test_np, preds_proba)

# # Run Optuna study
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=30)

# # Best result
# print("Best Trial:")
# print(study.best_trial)


In [7]:
# best_params = study.best_trial.params

# # Format optimizer
# best_params["optimizer_params"] = {"lr": best_params.pop("lr")}
# best_params["seed"] = 42
# best_params["verbose"] = 1

# # Train final model
# final_model = TabNetClassifier(**best_params, device_name='cuda' if torch.cuda.is_available() else 'cpu')

# final_model.fit(
#     X_resampled, y_resampled,
#     eval_set=[(X_test_np, y_test_np)],
#     eval_metric=['auc'],
#     max_epochs=200,
#     patience=20,
#     batch_size=1024,
#     virtual_batch_size=128,
#     drop_last=False
# )

# # Predict and evaluate
# final_preds = final_model.predict(X_test_np)
# final_probs = final_model.predict_proba(X_test_np)[:, 1]

In [8]:
# # Predictions
# y_pred_tune = final_model.predict(X_test_np)
# y_proba_tune = final_model.predict_proba(X_test_np)[:, 1]
# y_train_pred_tune = final_model.predict(X_train.values)

# # Evaluation
# print("Best Parameters:", study.best_params)
# print(confusion_matrix(y_test_np, y_pred_tune))
# print('------------------------------------------------------')
# print(classification_report(y_test_np, y_pred_tune, digits=3))
# print('------------------------------------------------------')
# print(f"Test Avg Precision: {average_precision_score(y_test_np, y_proba_tune):.3f}")
# print(f"Test AUC: {roc_auc_score(y_test_np, y_proba_tune):.3f}")
# print(f"Test Cohen's Kappa: {cohen_kappa_score(y_test_np, y_pred_tune):.3f}")
# print(f"Training recall: {recall_score(y_train, y_train_pred_tune):.3f}")

## Second

In [9]:
# # ✅ Transformer to convert Pandas → NumPy (TabNet requires NumPy)
# class ToNumpyTransformer(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         return self
#     def transform(self, X):
#         return X.values if hasattr(X, 'values') else X

# # ✅ Build the pipeline with SMOTE inside
# pipeline = Pipeline(steps=[
#     ('smote', SMOTE(random_state=42)),
#     ('to_numpy', ToNumpyTransformer()),
#     ('clf', TabNetClassifier(verbose=0, seed=42, device_name='cuda' if torch.cuda.is_available() else 'cpu'))
# ])

# # ✅ Optuna objective
# def objective(trial):
#     # TabNet hyperparameters
#     params = {
#         "n_d": trial.suggest_int("n_d", 8, 64),
#         "n_a": trial.suggest_int("n_a", 8, 64),
#         "n_steps": trial.suggest_int("n_steps", 3, 10),
#         "gamma": trial.suggest_float("gamma", 1.0, 2.0),
#         "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
#         "optimizer_params": {"lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True)},
#         "mask_type": trial.suggest_categorical("mask_type", ["entmax", "sparsemax"]),
#     }
    
#     # ✅ Set parameters on pipeline
#     pipeline.set_params(**{f"clf__{k}": v for k, v in params.items()})

#     # ✅ Fit TabNet via pipeline (SMOTE applies on the fly)
#     pipeline.fit(
#         X_train, y_train,
#         clf__eval_set=[(X_test.values, y_test.values)],
#         clf__eval_metric=['balanced_accuracy'],
#         clf__max_epochs=100,
#         clf__patience=10,
#         clf__batch_size=1024,
#         clf__virtual_batch_size=128,
#         clf__drop_last=False
#     )

#     # ✅ Predictions
#     preds_proba = pipeline.named_steps['clf'].predict_proba(X_test.values)[:, 1]
#     return roc_auc_score(y_test, preds_proba)

# # ✅ Optuna study
# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=20)

# # ✅ Best trial output
# print("Best Trial:")
# print(study.best_trial)


In [10]:
# # ✅ Get best params from Optuna
# best_params = study.best_trial.params

# # ✅ Fix optimizer format for TabNet
# best_params["optimizer_params"] = {"lr": best_params.pop("lr")}
# best_params["seed"] = 42
# best_params["verbose"] = 1

# # ✅ Build final pipeline with best params
# final_pipeline = Pipeline(steps=[
#     ('smote', SMOTE(random_state=42)),
#     ('to_numpy', ToNumpyTransformer()),
#     ('clf', TabNetClassifier(**best_params, device_name='cuda' if torch.cuda.is_available() else 'cpu'))
# ])

# # ✅ Fit pipeline (SMOTE applied automatically)
# final_pipeline.fit(
#     X_train, y_train,
#     clf__eval_set=[(X_test.values, y_test.values)],
#     clf__eval_metric=['balanced_accuracy'],
#     clf__max_epochs=200,
#     clf__patience=10,
#     clf__batch_size=1024,
#     clf__virtual_batch_size=128,
#     clf__drop_last=False
# )

# # ✅ Predictions
# final_preds = final_pipeline.named_steps['clf'].predict(X_test.values)
# final_probs = final_pipeline.named_steps['clf'].predict_proba(X_test.values)[:, 1]


In [11]:


# # ------------------------
# # ✅ Predictions
# # ------------------------
# y_pred_tune = final_pipeline.predict(X_test)   # pipeline handles SMOTE + TabNet
# y_proba_tune = final_pipeline.predict_proba(X_test)[:, 1]

# # 🔥 Training predictions should come from pipeline (not raw model)
# y_train_pred_tune = final_pipeline.predict(X_train)

# # ------------------------
# # ✅ Evaluation
# # ------------------------
# print("Best Parameters:", study.best_params)
# print(confusion_matrix(y_test, y_pred_tune))
# print('------------------------------------------------------')
# print(classification_report(y_test, y_pred_tune, digits=3))
# print('------------------------------------------------------')
# print(f"Test Avg Precision: {average_precision_score(y_test, y_proba_tune):.3f}")
# print(f"Test AUC: {roc_auc_score(y_test, y_proba_tune):.3f}")
# print(f"Test Cohen's Kappa: {cohen_kappa_score(y_test, y_pred_tune):.3f}")
# print(f"Training Recall: {recall_score(y_train, y_train_pred_tune):.3f}")


## Third

In [12]:
# # =====================================================
# # ✅ Final Streamlined TabNet Workflow with Optuna
# # =====================================================
# import pandas as pd
# import numpy as np
# import torch
# import optuna
# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (
#     classification_report, confusion_matrix, recall_score,
#     roc_auc_score, average_precision_score, cohen_kappa_score
# )
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline
# from sklearn.base import BaseEstimator, TransformerMixin
# from pytorch_tabnet.tab_model import TabNetClassifier

# import random

# # Set all seeds
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# torch.cuda.manual_seed(SEED)

# # For deterministic behavior in CUDA
# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False


# # =====================================================
# # 1️⃣ Data Preparation
# # =====================================================
# # Load data
# df = pd.read_csv(r"C:\Users\dbastola2022\OneDrive - Florida Atlantic University\Academics\Research\Malnutrition\MICS\malnutrition\Dataset\ch.csv")

# # Split features & target
# X = df.drop(columns=['status'])
# y = df['status']

# # Train-test split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=42
# )

# # =====================================================
# # 2️⃣ Transformer: Pandas → NumPy for TabNet
# # =====================================================
# class ToNumpyTransformer(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         return self
#     def transform(self, X):
#         return X.to_numpy(dtype=np.float32) if hasattr(X, 'to_numpy') else X.astype(np.float32)

# # =====================================================
# # 3️⃣ Pipeline: SMOTE → NumPy → TabNet
# # =====================================================
# pipeline = Pipeline(steps=[
#     ('smote', SMOTE(random_state=42)),
#     ('to_numpy', ToNumpyTransformer()),
#     ('clf', TabNetClassifier(verbose=0, seed=SEED, device_name='cuda' if torch.cuda.is_available() else 'cpu'))
# ])

# # =====================================================
# # 4️⃣ Optuna Objective Function
# # =====================================================
# def objective(trial):
#     # Suggest hyperparameters
#     params = {
#         "n_d": trial.suggest_int("n_d", 8, 64),
#         "n_a": trial.suggest_int("n_a", 8, 64),
#         "n_steps": trial.suggest_int("n_steps", 3, 10),
#         "gamma": trial.suggest_float("gamma", 1.0, 2.0),
#         "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
#         "optimizer_params": {"lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True)},
#         "mask_type": trial.suggest_categorical("mask_type", ["entmax", "sparsemax"]),
#     }

#     # Apply params to pipeline
#     pipeline.set_params(**{f"clf__{k}": v for k, v in params.items()})

#     # Fit model (SMOTE applied on-the-fly)
#     pipeline.fit(
#         X_train, y_train,
#         clf__eval_set=[(X_test.to_numpy(dtype=np.float32), y_test.to_numpy())],
#         clf__eval_metric=['balanced_accuracy'],
#         clf__max_epochs=100,
#         clf__patience=10,
#         clf__batch_size=1024,
#         clf__virtual_batch_size=128,
#         clf__drop_last=False
#     )

#     # Predict & compute AUC
#     preds_proba = pipeline.named_steps['clf'].predict_proba(X_test.to_numpy(dtype=np.float32))[:, 1]
#     return roc_auc_score(y_test, preds_proba)

# # =====================================================
# # 5️⃣ Run Optuna Tuning
# # =====================================================
# sampler = optuna.samplers.TPESampler(seed=SEED)
# study = optuna.create_study(direction="maximize", sampler=sampler)
# study.optimize(objective, n_trials=50)

# print("Best Trial:", study.best_trial)

# # Format best params for final model
# best_params = study.best_trial.params
# best_params["optimizer_params"] = {"lr": best_params.pop("lr")}
# best_params["seed"] = 42
# best_params["verbose"] = 1

# # =====================================================
# # 6️⃣ Final Pipeline with Best Params
# # =====================================================
# final_pipeline = Pipeline(steps=[
#     ('smote', SMOTE(random_state=42)),
#     ('to_numpy', ToNumpyTransformer()),
#     ('clf', TabNetClassifier(**best_params, device_name='cuda' if torch.cuda.is_available() else 'cpu'))
# ])

# # Fit final model
# final_pipeline.fit(
#     X_train, y_train,
#     clf__eval_set=[(X_test.to_numpy(dtype=np.float32), y_test.to_numpy())],
#     clf__eval_metric=['balanced_accuracy'],
#     clf__max_epochs=200,
#     clf__patience=10,
#     clf__batch_size=1024,
#     clf__virtual_batch_size=128,
#     clf__drop_last=False
# )




In [13]:
# # =====================================================
# # 7️⃣ Predictions & Evaluation
# # =====================================================
# y_pred = final_pipeline.predict(X_test)
# y_proba = final_pipeline.predict_proba(X_test)[:, 1]
# y_train_pred = final_pipeline.predict(X_train)

# print("\n✅ FINAL MODEL PERFORMANCE")
# print("Best Params:", study.best_params)
# print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
# print("------------------------------------------------------")
# print(classification_report(y_test, y_pred, digits=3))
# print("------------------------------------------------------")
# print(f"Test Avg Precision: {average_precision_score(y_test, y_proba):.3f}")
# print(f"Test AUC: {roc_auc_score(y_test, y_proba):.3f}")
# print(f"Test Cohen's Kappa: {cohen_kappa_score(y_test, y_pred):.3f}")
# print(f"Training Recall: {recall_score(y_train, y_train_pred):.3f}")

# Fourth 

In [14]:
# # =====================================================
# # ✅ Final TabNet + Optuna Hyperparameter Tuning (Clean Version)
# # =====================================================
# import pandas as pd
# import numpy as np
# import torch
# import random
# import optuna

# from sklearn.model_selection import train_test_split
# from sklearn.metrics import (
#     classification_report, confusion_matrix, recall_score,
#     roc_auc_score, average_precision_score, cohen_kappa_score
# )
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline
# from sklearn.base import BaseEstimator, TransformerMixin
# from pytorch_tabnet.tab_model import TabNetClassifier
# import torch.optim as optim

# # =====================================================
# # 🔒 Set all seeds for reproducibility
# # =====================================================
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# torch.cuda.manual_seed(SEED)

# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False

# # =====================================================
# # 1️⃣ Load & Split Data
# # =====================================================
# df = pd.read_csv(r"C:\Users\dbastola2022\OneDrive - Florida Atlantic University\Academics\Research\Malnutrition\MICS\malnutrition\Dataset\ch.csv")

# X = df.drop(columns=['status'])
# y = df['status']

# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=SEED
# )

# # =====================================================
# # 2️⃣ Transformer: Pandas → NumPy for TabNet
# # =====================================================
# class ToNumpyTransformer(BaseEstimator, TransformerMixin):
#     def fit(self, X, y=None):
#         return self
#     def transform(self, X):
#         return X.to_numpy(dtype=np.float32) if hasattr(X, 'to_numpy') else X.astype(np.float32)

# # =====================================================
# # 3️⃣ Pipeline: SMOTE → NumPy → TabNet
# # =====================================================
# pipeline = Pipeline(steps=[
#     ('smote', SMOTE(random_state=SEED)),
#     ('to_numpy', ToNumpyTransformer()),
#     ('clf', TabNetClassifier(verbose=0, seed=SEED,
#                              device_name='cuda' if torch.cuda.is_available() else 'cpu'))
# ])

# # =====================================================
# # 4️⃣ Optuna Objective Function
# # =====================================================
# def objective(trial):
#     # 🔥 TabNet model hyperparameters
#     params = {
#         "n_d": trial.suggest_int("n_d", 8, 64),
#         "n_a": trial.suggest_int("n_a", 8, 64),
#         "n_steps": trial.suggest_int("n_steps", 3, 10),
#         "gamma": trial.suggest_float("gamma", 1.0, 2.0),
#         "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
#         "optimizer_fn": trial.suggest_categorical("optimizer_fn", [optim.Adam, optim.AdamW, optim.RMSprop]),
#         "optimizer_params": {"lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True)},
#         "mask_type": trial.suggest_categorical("mask_type", ["entmax", "sparsemax"]),
#         "n_shared": trial.suggest_int("n_shared", 1, 3),
#         "n_independent": trial.suggest_int("n_independent", 1, 3),
#     }

#     # ✅ Fit-time hyperparameters (NOT passed to model constructor)
#     batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
#     virtual_batch_size = trial.suggest_categorical("virtual_batch_size", [64, 128])

#     # ✅ Apply model params to pipeline
#     pipeline.set_params(**{f"clf__{k}": v for k, v in params.items()})

#     # ✅ Fit model with trial-specific batch sizes
#     pipeline.fit(
#         X_train, y_train,
#         clf__eval_set=[(X_test.to_numpy(dtype=np.float32), y_test.to_numpy())],
#         clf__eval_metric=['balanced_accuracy'],
#         clf__max_epochs=100,
#         clf__patience=10,
#         clf__batch_size=batch_size,
#         clf__virtual_batch_size=virtual_batch_size,
#         clf__drop_last=False,
#         clf__num_workers=0
#     )

#     # ✅ Evaluate using ROC-AUC
#     preds_proba = pipeline.named_steps['clf'].predict_proba(X_test.to_numpy(dtype=np.float32))[:, 1]
#     return roc_auc_score(y_test, preds_proba)

# # =====================================================
# # 5️⃣ Run Optuna Study
# # =====================================================
# sampler = optuna.samplers.TPESampler(seed=SEED)
# study = optuna.create_study(direction="maximize", sampler=sampler)
# study.optimize(objective, n_trials=50)

# print("✅ Best Trial:", study.best_trial)

# # =====================================================
# # 6️⃣ Extract Best Params for Final Model
# # =====================================================
# best_params = study.best_trial.params

# # Pull out fit-time params
# best_batch_size = best_params.pop("batch_size")
# best_virtual_batch_size = best_params.pop("virtual_batch_size")

# best_params["optimizer_params"] = {"lr": best_params.pop("lr")}
# best_params["seed"] = SEED
# best_params["verbose"] = 1

# # =====================================================
# # 7️⃣ Final Model Training with Best Params
# # =====================================================
# final_pipeline = Pipeline(steps=[
#     ('smote', SMOTE(random_state=SEED)),
#     ('to_numpy', ToNumpyTransformer()),
#     ('clf', TabNetClassifier(**best_params,
#                              device_name='cuda' if torch.cuda.is_available() else 'cpu'))
# ])

# final_pipeline.fit(
#     X_train, y_train,
#     clf__eval_set=[(X_test.to_numpy(dtype=np.float32), y_test.to_numpy())],
#     clf__eval_metric=['balanced_accuracy'],
#     clf__max_epochs=200,
#     clf__patience=10,
#     clf__batch_size=best_batch_size,
#     clf__virtual_batch_size=best_virtual_batch_size,
#     clf__drop_last=False,
#     clf__num_workers=0
# )

# # =====================================================
# # 8️⃣ Evaluate Final Model
# # =====================================================
# y_pred = final_pipeline.predict(X_test)
# y_proba = final_pipeline.predict_proba(X_test)[:, 1]
# y_train_pred = final_pipeline.predict(X_train)

# print("\n✅ FINAL MODEL PERFORMANCE")
# print("Best Params:", study.best_params)
# print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
# print("------------------------------------------------------")
# print(classification_report(y_test, y_pred, digits=3))
# print("------------------------------------------------------")
# print(f"Test Avg Precision: {average_precision_score(y_test, y_proba):.3f}")
# print(f"Test AUC: {roc_auc_score(y_test, y_proba):.3f}")
# print(f"Test Cohen's Kappa: {cohen_kappa_score(y_test, y_pred):.3f}")
# print(f"Training Recall: {recall_score(y_train, y_train_pred):.3f}")


## Final TabNet + Optuna + StratifiedKFold CV (Robust)

In [15]:
# # =====================================================
# # ✅ Final TabNet + Optuna + StratifiedKFold CV (Robust)
# # =====================================================
# import pandas as pd
# import numpy as np
# import torch
# import random
# import optuna

# from sklearn.model_selection import StratifiedKFold, train_test_split
# from sklearn.metrics import (
#     classification_report, confusion_matrix, recall_score,
#     roc_auc_score, average_precision_score, cohen_kappa_score
# )
# from imblearn.over_sampling import SMOTE
# from pytorch_tabnet.tab_model import TabNetClassifier
# import torch.optim as optim

# # =====================================================
# # 🔒 Set all seeds for reproducibility
# # =====================================================
# SEED = 42
# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# torch.cuda.manual_seed(SEED)

# torch.backends.cudnn.deterministic = True
# torch.backends.cudnn.benchmark = False

# # =====================================================
# # 1️⃣ Load & Split Data (Hold-Out Test for Final Eval)
# # =====================================================
# df = pd.read_csv(r"C:\Users\dbastola2022\OneDrive - Florida Atlantic University\Academics\Research\Malnutrition\MICS\malnutrition\Dataset\ch.csv")

# X = df.drop(columns=['status'])
# y = df['status']

# # Keep 20% for FINAL testing (not used during Optuna CV)
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.2, stratify=y, random_state=SEED
# )

# # =====================================================
# # 2️⃣ Optuna Objective with Stratified K-Fold
# # =====================================================
# def objective(trial):
#     # 🔥 TabNet model hyperparameters
#     params = {
#         "n_d": trial.suggest_int("n_d", 8, 64),
#         "n_a": trial.suggest_int("n_a", 8, 64),
#         "n_steps": trial.suggest_int("n_steps", 3, 10),
#         "gamma": trial.suggest_float("gamma", 1.0, 2.0),
#         "lambda_sparse": trial.suggest_float("lambda_sparse", 1e-6, 1e-2, log=True),
#         "optimizer_fn": trial.suggest_categorical("optimizer_fn", [optim.Adam, optim.AdamW, optim.RMSprop]),
#         "optimizer_params": {"lr": trial.suggest_float("lr", 1e-4, 1e-2, log=True)},
#         "mask_type": trial.suggest_categorical("mask_type", ["entmax", "sparsemax"]),
#         "n_shared": trial.suggest_int("n_shared", 1, 3),
#         "n_independent": trial.suggest_int("n_independent", 1, 3),
#         "seed": SEED,
#         "verbose": 0,
#         "device_name": 'cuda' if torch.cuda.is_available() else 'cpu'
#     }

#     # ✅ Fit-time hyperparameters (not constructor args)
#     batch_size = trial.suggest_categorical("batch_size", [256, 512, 1024])
#     virtual_batch_size = trial.suggest_categorical("virtual_batch_size", [64, 128])

#     # ✅ Stratified K-Fold CV
#     skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
#     auc_scores = []

#     for train_idx, val_idx in skf.split(X_train, y_train):
#         X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
#         y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

#         # ✅ Apply SMOTE to training fold only
#         smote = SMOTE(random_state=SEED)
#         X_res, y_res = smote.fit_resample(X_tr, y_tr)

#         # ✅ Convert to NumPy for TabNet
#         X_res_np = X_res.to_numpy(dtype=np.float32)
#         y_res_np = y_res.to_numpy().astype(np.int64)
#         X_val_np = X_val.to_numpy(dtype=np.float32)
#         y_val_np = y_val.to_numpy().astype(np.int64)

#         # ✅ Initialize TabNet model for this fold
#         model = TabNetClassifier(**params)

#         # ✅ Train model
#         model.fit(
#             X_res_np, y_res_np,
#             eval_set=[(X_val_np, y_val_np)],
#             eval_metric=['balanced_accuracy'],
#             max_epochs=100,
#             patience=10,
#             batch_size=batch_size,
#             virtual_batch_size=virtual_batch_size,
#             drop_last=False,
#             num_workers=0
#         )

#         # ✅ Predict & compute AUC for this fold
#         preds_proba = model.predict_proba(X_val_np)[:, 1]
#         auc_scores.append(roc_auc_score(y_val_np, preds_proba))

#     # ✅ Return mean AUC across folds
#     return np.mean(auc_scores)

# # =====================================================
# # 3️⃣ Run Optuna Study
# # =====================================================
# sampler = optuna.samplers.TPESampler(seed=SEED)
# study = optuna.create_study(direction="maximize", sampler=sampler)
# study.optimize(objective, n_trials=50)

# print("✅ Best Trial:", study.best_trial)

# # =====================================================
# # 4️⃣ Train Final Model on Full Train Set w/ Best Params
# # =====================================================
# best_params = study.best_trial.params

# # Extract fit-time params
# best_batch_size = best_params.pop("batch_size")
# best_virtual_batch_size = best_params.pop("virtual_batch_size")

# best_params["optimizer_params"] = {"lr": best_params.pop("lr")}
# best_params["seed"] = SEED
# best_params["verbose"] = 1
# best_params["device_name"] = 'cuda' if torch.cuda.is_available() else 'cpu'

# # ✅ Apply SMOTE to the ENTIRE training set
# smote = SMOTE(random_state=SEED)
# X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# X_train_np = X_train_res.to_numpy(dtype=np.float32)
# y_train_np = y_train_res.to_numpy().astype(np.int64)
# X_test_np = X_test.to_numpy(dtype=np.float32)
# y_test_np = y_test.to_numpy().astype(np.int64)

# # ✅ Train final model
# final_model = TabNetClassifier(**best_params)
# final_model.fit(
#     X_train_np, y_train_np,
#     eval_set=[(X_test_np, y_test_np)],
#     eval_metric=['balanced_accuracy'],
#     max_epochs=200,
#     patience=10,
#     batch_size=best_batch_size,
#     virtual_batch_size=best_virtual_batch_size,
#     drop_last=False,
#     num_workers=0
# )

In [16]:
# # =====================================================
# # 5️⃣ Final Evaluation on Hold-Out Test Set
# # =====================================================
# y_pred = final_model.predict(X_test_np)
# y_proba = final_model.predict_proba(X_test_np)[:, 1]

# print("\n✅ FINAL MODEL PERFORMANCE")
# print("Best Params:", study.best_params)
# print("Confusion Matrix:\n", confusion_matrix(y_test_np, y_pred))
# print("------------------------------------------------------")
# print(classification_report(y_test_np, y_pred, digits=3))
# print("------------------------------------------------------")
# print(f"Test Avg Precision: {average_precision_score(y_test_np, y_proba):.3f}")
# print(f"Test AUC: {roc_auc_score(y_test_np, y_proba):.3f}")
# print(f"Test Cohen's Kappa: {cohen_kappa_score(y_test_np, y_pred):.3f}")
# print(f"Test Recall: {recall_score(y_test_np, y_pred):.3f}")

# From the parameter receive from final model

In [1]:
import pandas as pd
import numpy as np
import torch
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from pytorch_tabnet.tab_model import TabNetClassifier
import torch.optim as optim
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, recall_score, cohen_kappa_score

# Your best hyperparameters
best_params = {
    'n_d': 23,
    'n_a': 61,
    'n_steps': 3,
    'gamma': 1.5573224296071704,
    'lambda_sparse': 2.6420767368057722e-05,
    'optimizer_fn': optim.RMSprop,
    'optimizer_params': {'lr': 0.006958832760587134},
    'mask_type': 'sparsemax',
    'n_shared': 3,
    'n_independent': 2,
    'seed': 42,
    'verbose': 1,
}

batch_size = 256
virtual_batch_size = 128
SEED = 42

# Set seeds for reproducibility
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Data loading example (adjust path)
df = pd.read_csv(r"C:\Users\dbastola2022\OneDrive - Florida Atlantic University\Academics\Research\Malnutrition\MICS\malnutrition\Dataset\ch.csv")

X = df.drop(columns=['status'])
y = df['status']

# Simple train-test split (you can replace with your own split)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

# Transformer: convert pandas DataFrame to numpy float32 (required by TabNet)
class ToNumpyTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X.to_numpy(dtype=np.float32) if hasattr(X, 'to_numpy') else X.astype(np.float32)

# Build pipeline with SMOTE + transformer + TabNetClassifier
pipeline = Pipeline(steps=[
    ('smote', SMOTE(random_state=SEED)),
    ('to_numpy', ToNumpyTransformer()),
    ('clf', TabNetClassifier(**best_params,
                             device_name='cuda' if torch.cuda.is_available() else 'cpu'))
])

# Fit the model
pipeline.fit(
    X_train, y_train,
    clf__eval_set=[(X_test.to_numpy(dtype=np.float32), y_test.to_numpy())],
    clf__eval_metric=['balanced_accuracy'],
    clf__max_epochs=200,
    clf__patience=10,
    clf__batch_size=batch_size,
    clf__virtual_batch_size=virtual_batch_size,
    clf__drop_last=False,
    clf__num_workers=0
)

# Predict & Evaluate
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=3))
print(f"AUC: {roc_auc_score(y_test, y_proba):.3f}")
print(f"Average Precision: {average_precision_score(y_test, y_proba):.3f}")
print(f"Cohen's Kappa: {cohen_kappa_score(y_test, y_pred):.3f}")
print(f"Recall (Train): {recall_score(y_train, pipeline.predict(X_train)):.3f}")

c:\Users\dbastola2022\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 0.84035 | val_0_balanced_accuracy: 0.63258 |  0:00:00s
epoch 1  | loss: 0.66474 | val_0_balanced_accuracy: 0.57466 |  0:00:01s
epoch 2  | loss: 0.63821 | val_0_balanced_accuracy: 0.60968 |  0:00:02s
epoch 3  | loss: 0.6254  | val_0_balanced_accuracy: 0.63667 |  0:00:03s
epoch 4  | loss: 0.60889 | val_0_balanced_accuracy: 0.61773 |  0:00:03s
epoch 5  | loss: 0.59138 | val_0_balanced_accuracy: 0.62859 |  0:00:04s
epoch 6  | loss: 0.59009 | val_0_balanced_accuracy: 0.63159 |  0:00:05s
epoch 7  | loss: 0.57757 | val_0_balanced_accuracy: 0.67091 |  0:00:06s
epoch 8  | loss: 0.55989 | val_0_balanced_accuracy: 0.65986 |  0:00:06s
epoch 9  | loss: 0.55975 | val_0_balanced_accuracy: 0.66949 |  0:00:07s
epoch 10 | loss: 0.55179 | val_0_balanced_accuracy: 0.6741  |  0:00:08s
epoch 11 | loss: 0.54159 | val_0_balanced_accuracy: 0.70486 |  0:00:09s
epoch 12 | loss: 0.53477 | val_0_balanced_accuracy: 0.70667 |  0:00:09s
epoch 13 | loss: 0.52889 | val_0_balanced_accuracy: 0.69751 |  0

c:\Users\dbastola2022\AppData\Local\Programs\Python\Python310\lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Confusion Matrix:
 [[520 187]
 [137 467]]

Classification Report:
               precision    recall  f1-score   support

           0      0.791     0.736     0.762       707
           1      0.714     0.773     0.742       604

    accuracy                          0.753      1311
   macro avg      0.753     0.754     0.752      1311
weighted avg      0.756     0.753     0.753      1311

AUC: 0.826
Average Precision: 0.825
Cohen's Kappa: 0.506
Recall (Train): 0.757
